# Lab 03 · Architecture tools and our own MCP server

**Prepared by Mustafa Assaf** · 15–20 minute lab

## Give the program tools through MCP

**So far:** the model only read text that we put into the prompt.

**New:** *tools*, which are functions that read data or take an action. The **Model Context Protocol (MCP)** publishes tools so that any MCP client can discover and call them, with typed arguments and clear errors.

We build an MCP server with two tools. `get_project` *reads* the facts of one project. `save_decision` *changes state* by recording a decision, and it refuses bad input. The official MCP SDK connects the client and server inside this notebook, so no port, external service, or model key is needed.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. This lab needs no model key.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'mcp']):
    %pip install -q "agentic-ai-coursekit[mcp] @ https://github.com/mkassaf/agentic-ai-course/archive/dbe84948102e503e581fd5236bb7e67e63dba56c.zip"

## Step 1 · A server with a read tool

`@server.tool(...)` registers a normal Python function as an MCP tool. The SDK reads the type hints (`name: str`) to build an input schema for clients. When a request cannot be served, the tool raises `ToolError` with a message for the caller. The SDK then sends back an error result instead of crashing.

In [ ]:
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

PROJECTS = {
    "Course portal":     {"team_size": 3,  "independent_deploys": False},
    "Payments platform": {"team_size": 12, "independent_deploys": True},
}
server = MCPServer("architecture-tools")

@server.tool(description="Read the team size and deployment facts of one project.")
def get_project(name: str) -> dict:
    if name not in PROJECTS:
        raise ToolError(f"Unknown project {name!r}. Known projects: {sorted(PROJECTS)}")
    return PROJECTS[name]

print(get_project("Course portal"))  # the decorator leaves a normal Python function

## Step 2 · A write tool that validates its input

A tool that changes state needs stricter checks than a tool that only reads. `save_decision` accepts only a known project and one of our two labels. It records the decision in the `decisions` dictionary, which stands in for a real database.

In [ ]:
LABELS = ("modular monolith", "microservices")
decisions = {}  # the state that save_decision changes

@server.tool(description="Record the architecture decision for one project.")
def save_decision(project: str, choice: str) -> str:
    if project not in PROJECTS:
        raise ToolError(f"Unknown project {project!r}")
    if choice not in LABELS:
        raise ToolError(f"choice must be one of {LABELS}")
    decisions[project] = choice
    return f"Saved {choice!r} for {project}"

## Step 3 · Discover the tools as a client

The client does not read our Python code. It asks the server for its tools and gets back each tool's name, description, and input schema. Lab 04's model needs exactly this information to choose a tool. `await` is needed because MCP messages are asynchronous, and Colab notebooks accept `await` directly.

In [ ]:
from mcp import Client

async with Client(server) as client:
    listing = await client.list_tools()
for tool in listing.tools:
    args = ", ".join(f"{name}: {spec['type']}" for name, spec in tool.input_schema["properties"].items())
    print(f"{tool.name}({args})\n    {tool.description}")

## Step 4 · Call the tools in a fixed workflow

For each project, our code reads the facts, applies the policy in Python, and saves the result. A dictionary result comes back as JSON text, so `json.loads` turns it into a dictionary again. The order of the calls is fixed by our code. In Lab 04 the model chooses the order.

In [ ]:
import json

async with Client(server) as client:
    for name in PROJECTS:
        result = await client.call_tool("get_project", {"name": name})
        facts = json.loads(result.content[0].text)
        both = facts["team_size"] >= 8 and facts["independent_deploys"]
        choice = "microservices" if both else "modular monolith"
        saved = await client.call_tool("save_decision", {"project": name, "choice": choice})
        print(saved.content[0].text)
print("Server state:", decisions)
assert decisions == {"Course portal": "modular monolith", "Payments platform": "microservices"}

## Step 5 · See how tools report errors

A client must handle failures. Each bad call below returns a result with `is_error=True` and a message, instead of raising an exception in the client. The SDK also checks arguments against the input schema, so a missing argument is rejected before our function runs. We print only the first line of each message. Rejected calls must not change the saved state.

In [ ]:
bad_calls = [("get_project", {"name": "Billing"}),
             ("save_decision", {"project": "Course portal", "choice": "serverless"}),
             ("save_decision", {"project": "Course portal"})]  # missing the choice argument
async with Client(server) as client:
    for tool, arguments in bad_calls:
        result = await client.call_tool(tool, arguments)
        print(f"is_error={result.is_error}: {result.content[0].text.splitlines()[0]}")
assert decisions["Course portal"] == "modular monolith", "A rejected call must not change state."

## What you should see

Step 1 prints the Course portal's facts. Step 3 lists `get_project(name: string)` and `save_decision(project: string, choice: string)` with their descriptions. Step 4 prints two `Saved ...` lines and the server state with both decisions. Step 5 prints three lines that start with `is_error=True`. The server may also log each failure on a separate line. That log is the server's view of the same errors.

## Try it

1. Add `"Internal wiki": {"team_size": 9, "independent_deploys": False}` to `PROJECTS`. Run Steps 1 to 4 again, and update the final `assert` to include it.
2. Add a third tool, `list_projects() -> list`, that returns `sorted(PROJECTS)`. Run Step 3 and find it in the listing.
3. In Step 5, change `"Billing"` to the number `42`. Read how the SDK reports an argument of the wrong type.

## Takeaway

MCP separates *what a tool offers* (a name, a description, and a schema) from *who calls it*. Tools validate their own input, and errors come back as data that a client, or a model, can read and act on. Lab 04 lets the model decide which tool to call next.

## Read further

- [Official SDK client guide](https://py.sdk.modelcontextprotocol.io/client/)
- [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture)